# HPC Integration - dask_setup Recipe

This notebook demonstrates how to integrate dask_setup with High Performance Computing (HPC) schedulers like PBS and SLURM, including automatic job script generation, resource detection, SSH tunnel setup, and best practices for running Dask on HPC systems.

## Key Learning Points
- Automatic PBS/SLURM job script generation
- Resource detection from scheduler environment variables
- Dashboard access via SSH tunneling
- HPC-specific configuration optimization
- Job monitoring and status checking
- Best practices for HPC scientific computing workflows

## Setup and Imports

Import necessary libraries and configure the environment for HPC integration.

In [ ]:
import os
import sys
import socket
import time
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Add dask_setup to path
sys.path.insert(0, str(Path.cwd().parent.parent.parent / "src"))
sys.path.insert(0, str(Path.cwd().parent))

# Core imports
import dask
import dask.array as da
import numpy as np

# Import dask_setup
try:
    from dask_setup import setup_dask_client
    print("✅ dask_setup imported successfully")
except ImportError as e:
    print(f"❌ Failed to import dask_setup: {e}")
    
# System monitoring imports
try:
    import psutil
    PSUTIL_AVAILABLE = True
    print("✅ psutil available for system monitoring")
except ImportError:
    PSUTIL_AVAILABLE = False
    print("⚠️  psutil not available - system monitoring limited")

# Utility imports
try:
    from utils import format_duration, timer
    print("✅ utils imported successfully")
except ImportError as e:
    print(f"⚠️  utils not available: {e}")
    # Define minimal versions
    def format_duration(seconds):
        return f"{seconds:.2f}s"
    from contextlib import contextmanager
    @contextmanager
    def timer(name):
        start = time.time()
        result = {'total': 0}
        yield result
        result['total'] = time.time() - start

print("\n🚀 Setup complete!")

## HPC Environment Detection

Let's detect what HPC environment we're running in and what resources are available.

In [ ]:
def detect_hpc_environment():
    """Detect HPC scheduler environment and available resources."""
    
    env_info = {
        'scheduler': None,
        'job_id': None,
        'hostname': socket.gethostname(),
        'detected_resources': {},
        'environment_variables': {},
        'recommendations': []
    }
    
    # Check for PBS environment variables
    pbs_vars = {
        'PBS_JOBID': os.environ.get('PBS_JOBID'),
        'PBS_JOBFS': os.environ.get('PBS_JOBFS'),
        'PBS_NCPUS': os.environ.get('PBS_NCPUS'),
        'NCPUS': os.environ.get('NCPUS'),
        'PBS_MEM': os.environ.get('PBS_MEM'),
        'PBS_VMEM': os.environ.get('PBS_VMEM'),
        'PBS_QUEUE': os.environ.get('PBS_QUEUE'),
        'PBS_NODEFILE': os.environ.get('PBS_NODEFILE')
    }
    
    # Check for SLURM environment variables
    slurm_vars = {
        'SLURM_JOB_ID': os.environ.get('SLURM_JOB_ID'),
        'SLURM_CPUS_ON_NODE': os.environ.get('SLURM_CPUS_ON_NODE'),
        'SLURM_MEM_PER_NODE': os.environ.get('SLURM_MEM_PER_NODE'),
        'SLURM_MEM_PER_CPU': os.environ.get('SLURM_MEM_PER_CPU'),
        'SLURM_PARTITION': os.environ.get('SLURM_PARTITION'),
        'SLURM_NODELIST': os.environ.get('SLURM_NODELIST')
    }
    
    # Determine scheduler type and parse resources
    if any(pbs_vars.values()):
        env_info['scheduler'] = 'PBS'
        env_info['job_id'] = pbs_vars['PBS_JOBID']
        env_info['environment_variables'] = {k: v for k, v in pbs_vars.items() if v}
        
        # Parse PBS resources
        if pbs_vars['NCPUS'] or pbs_vars['PBS_NCPUS']:
            ncpus = int(pbs_vars['NCPUS'] or pbs_vars['PBS_NCPUS'])
            env_info['detected_resources']['cpus'] = ncpus
            
        if pbs_vars['PBS_MEM'] or pbs_vars['PBS_VMEM']:
            mem_str = pbs_vars['PBS_MEM'] or pbs_vars['PBS_VMEM']
            # Parse memory (e.g., "300gb", "64000mb")
            if 'gb' in mem_str.lower():
                memory_gb = float(mem_str.lower().replace('gb', ''))
            elif 'mb' in mem_str.lower():
                memory_gb = float(mem_str.lower().replace('mb', '')) / 1024
            else:
                memory_gb = None
            if memory_gb:
                env_info['detected_resources']['memory_gb'] = memory_gb
                
    elif any(slurm_vars.values()):
        env_info['scheduler'] = 'SLURM'
        env_info['job_id'] = slurm_vars['SLURM_JOB_ID']
        env_info['environment_variables'] = {k: v for k, v in slurm_vars.items() if v}
        
        # Parse SLURM resources
        if slurm_vars['SLURM_CPUS_ON_NODE']:
            env_info['detected_resources']['cpus'] = int(slurm_vars['SLURM_CPUS_ON_NODE'])
            
        if slurm_vars['SLURM_MEM_PER_NODE']:
            memory_mb = int(slurm_vars['SLURM_MEM_PER_NODE'])
            env_info['detected_resources']['memory_gb'] = memory_mb / 1024
        elif slurm_vars['SLURM_MEM_PER_CPU'] and env_info['detected_resources'].get('cpus'):
            memory_per_cpu_mb = int(slurm_vars['SLURM_MEM_PER_CPU'])
            total_memory_gb = (memory_per_cpu_mb * env_info['detected_resources']['cpus']) / 1024
            env_info['detected_resources']['memory_gb'] = total_memory_gb
    
    else:
        env_info['scheduler'] = 'Local/Unknown'
        env_info['recommendations'].append(
            "Not running under PBS or SLURM scheduler - using local system resources"
        )
    
    # Get system resources as fallback
    if PSUTIL_AVAILABLE:
        try:
            system_cpus = psutil.cpu_count(logical=False)
            system_memory_gb = psutil.virtual_memory().total / (1024**3)
            
            if 'cpus' not in env_info['detected_resources']:
                env_info['detected_resources']['cpus'] = system_cpus
                env_info['recommendations'].append(
                    f"Using system CPU count ({system_cpus}) - consider requesting specific resources in job script"
                )
                
            if 'memory_gb' not in env_info['detected_resources']:
                env_info['detected_resources']['memory_gb'] = system_memory_gb
                env_info['recommendations'].append(
                    f"Using system memory ({system_memory_gb:.1f} GB) - consider requesting specific resources in job script"
                )
        except Exception as e:
            env_info['recommendations'].append(f"Could not detect system resources: {e}")
    
    return env_info


# Detect the current environment
print("🔍 HPC ENVIRONMENT DETECTION")
print("=" * 50)

hpc_env = detect_hpc_environment()

print(f"Scheduler Type: {hpc_env['scheduler']}")
print(f"Hostname: {hpc_env['hostname']}")

if hpc_env['job_id']:
    print(f"Job ID: {hpc_env['job_id']}")

if hpc_env['detected_resources']:
    resources = hpc_env['detected_resources']
    print(f"\n📊 Detected Resources:")
    print(f"   CPUs: {resources.get('cpus', 'Unknown')}")
    print(f"   Memory: {resources.get('memory_gb', 'Unknown')} GB")

if hpc_env['environment_variables']:
    print(f"\n🌍 Environment Variables:")
    for var, value in hpc_env['environment_variables'].items():
        print(f"   {var}: {value}")

if hpc_env['recommendations']:
    print(f"\n💡 Recommendations:")
    for rec in hpc_env['recommendations']:
        print(f"   • {rec}")

print("\n✅ Environment detection complete")

## Job Script Generation

Let's create functions to generate PBS and SLURM job scripts optimized for dask_setup.

In [ ]:
def generate_pbs_script(
    cpus: int = 48,
    memory_gb: int = 190,
    walltime: str = "4:00:00",
    queue: str = "normalsr",
    jobfs_gb: int = 100,
    storage_paths: List[str] = None,
    script_name: str = "dask_job",
    python_script: str = "my_analysis.py"
) -> str:
    """Generate a PBS job script optimized for dask_setup."""
    
    if storage_paths is None:
        storage_paths = ["gdata/hh5"]
    
    storage_str = "+".join(storage_paths)
    project = os.environ.get('PROJECT', 'your_project')
    
    script = f"""#!/bin/bash
#PBS -P {project}
#PBS -q {queue}
#PBS -l ncpus={cpus}
#PBS -l mem={memory_gb}gb
#PBS -l jobfs={jobfs_gb}gb
#PBS -l walltime={walltime}
#PBS -l storage={storage_str}
#PBS -l wd
#PBS -N {script_name}
#PBS -o {script_name}.out
#PBS -e {script_name}.err

# Load required modules (adjust for your system)
module use /g/data/hh5/public/modules/
module load conda_concept/analysis3-unstable

# Set up environment for dask_setup
export TMPDIR="$PBS_JOBFS"  # Use fast local storage
export OMP_NUM_THREADS=1    # Prevent thread oversubscription

# Print job information
echo "=== PBS Job Information ==="
echo "Job ID: $PBS_JOBID"
echo "Queue: $PBS_QUEUE"  
echo "Nodes: $(cat $PBS_NODEFILE | sort | uniq | wc -l)"
echo "CPUs: $PBS_NCPUS"
echo "Memory: {memory_gb}GB"
echo "JobFS: $PBS_JOBFS ({jobfs_gb}GB)"
echo "Working directory: $(pwd)"
echo "Started at: $(date)"
echo "============================="

# Function to print resource usage
print_resource_usage() {{
    echo "\n=== Resource Usage ==="
    echo "Memory usage: $(free -h | awk '/^Mem:/ {{print $3 "/" $2 " (" $3/$2*100 "%)"}}')"
    echo "JobFS usage: $(df -h $PBS_JOBFS | tail -1 | awk '{{print $3 "/" $2 " (" $5 ")"}}')"
    echo "CPU usage: $(top -bn1 | grep "Cpu(s)" | sed "s/.*, *\\([0-9.]*\\)%* id.*/\\1/" | awk '{{print 100 - $1"%"}}')"
    echo "======================\n"
}}

# Set up signal handlers for monitoring
trap print_resource_usage EXIT
trap 'echo "\nJob interrupted at $(date)"; print_resource_usage; exit 130' INT TERM

# Run the Python script
echo "Starting Python analysis: {python_script}"
python {python_script}
exit_code=$?

# Print final status
if [ $exit_code -eq 0 ]; then
    echo "\n✅ Analysis completed successfully at $(date)"
else
    echo "\n❌ Analysis failed with exit code $exit_code at $(date)"
fi

# Final resource usage
print_resource_usage

# dask_setup automatically cleans up temporary files
# PBS automatically cleans up $PBS_JOBFS

exit $exit_code
"""
    
    return script


def generate_slurm_script(
    cpus: int = 48,
    memory_gb: int = 190,
    walltime: str = "4:00:00",
    partition: str = "normal",
    script_name: str = "dask_job",
    python_script: str = "my_analysis.py"
) -> str:
    """Generate a SLURM job script optimized for dask_setup."""
    
    script = f"""#!/bin/bash
#SBATCH --job-name={script_name}
#SBATCH --partition={partition}
#SBATCH --ntasks=1
#SBATCH --cpus-per-task={cpus}
#SBATCH --mem={memory_gb}G
#SBATCH --time={walltime}
#SBATCH --output={script_name}_%j.out
#SBATCH --error={script_name}_%j.err

# Load required modules (adjust for your system)
module load python/3.11
module load dask

# Set up environment for dask_setup
export TMPDIR="/tmp/slurm_$SLURM_JOB_ID"
mkdir -p $TMPDIR
export OMP_NUM_THREADS=1  # Prevent thread oversubscription

# Print job information
echo "=== SLURM Job Information ==="
echo "Job ID: $SLURM_JOB_ID"
echo "Partition: $SLURM_JOB_PARTITION"
echo "Node: $SLURMD_NODENAME"
echo "CPUs: $SLURM_CPUS_ON_NODE"
echo "Memory: {memory_gb}GB"
echo "Temp dir: $TMPDIR"
echo "Working directory: $(pwd)"
echo "Started at: $(date)"
echo "=============================="

# Function to print resource usage
print_resource_usage() {{
    echo "\n=== Resource Usage ==="
    if command -v sstat > /dev/null 2>&1; then
        echo "SLURM stats:"
        sstat -j $SLURM_JOB_ID --format=AveCPU,AveRSS,MaxRSS 2>/dev/null || echo "  sstat not available"
    fi
    echo "Memory: $(free -h | awk '/^Mem:/ {{print $3 "/" $2 " (" $3/$2*100 "%)"}}')"
    echo "Temp usage: $(du -sh $TMPDIR 2>/dev/null || echo 'N/A')"
    echo "======================\n"
}}

# Set up signal handlers
trap print_resource_usage EXIT
trap 'echo "\nJob interrupted at $(date)"; print_resource_usage; exit 130' INT TERM

# Run the Python script
echo "Starting Python analysis: {python_script}"
python {python_script}
exit_code=$?

# Print final status
if [ $exit_code -eq 0 ]; then
    echo "\n✅ Analysis completed successfully at $(date)"
else
    echo "\n❌ Analysis failed with exit code $exit_code at $(date)"
fi

# Final resource usage
print_resource_usage

# Clean up temporary directory
rm -rf $TMPDIR

exit $exit_code
"""
    
    return script


# Generate example scripts based on detected resources
detected_cpus = hpc_env['detected_resources'].get('cpus', 48)
detected_memory = int(hpc_env['detected_resources'].get('memory_gb', 190))

print("📝 GENERATING EXAMPLE JOB SCRIPTS")
print("=" * 50)
print(f"Using resources: {detected_cpus} CPUs, {detected_memory} GB memory")

# Generate PBS script
pbs_script = generate_pbs_script(
    cpus=detected_cpus,
    memory_gb=detected_memory,
    walltime="4:00:00",
    script_name="dask_analysis",
    python_script="analysis.py"
)

print("\n🐚 PBS Script Generated:")
print("─" * 30)
print(pbs_script[:500] + "..." if len(pbs_script) > 500 else pbs_script)

# Generate SLURM script
slurm_script = generate_slurm_script(
    cpus=detected_cpus,
    memory_gb=detected_memory,
    walltime="4:00:00",
    script_name="dask_analysis",
    python_script="analysis.py"
)

print("\n🔰 SLURM Script Generated:")
print("─" * 30)
print(slurm_script[:500] + "..." if len(slurm_script) > 500 else slurm_script)

print("\n✅ Job scripts generated successfully!")

## Save Generated Scripts

Let's save the generated scripts to files that can be used for job submission.

In [ ]:
# Create output directory
output_dir = Path("./hpc_scripts")
output_dir.mkdir(exist_ok=True)

print("💾 SAVING JOB SCRIPTS")
print("=" * 30)

# Save PBS script
pbs_file = output_dir / "dask_analysis.pbs"
with open(pbs_file, 'w') as f:
    f.write(pbs_script)
pbs_file.chmod(0o755)  # Make executable
print(f"📄 PBS script saved: {pbs_file}")

# Save SLURM script
slurm_file = output_dir / "dask_analysis.slurm"
with open(slurm_file, 'w') as f:
    f.write(slurm_script)
slurm_file.chmod(0o755)  # Make executable
print(f"📄 SLURM script saved: {slurm_file}")

# Generate example analysis script
analysis_script = '''#!/usr/bin/env python
"""Example dask_setup analysis script for HPC systems."""

import sys
import time
from pathlib import Path

import dask.array as da
import numpy as np
import xarray as xr

from dask_setup import setup_dask_client

def main():
    """Main analysis function."""
    print("🔬 Starting HPC scientific analysis with dask_setup...")
    
    # Setup Dask cluster (automatically detects PBS/SLURM resources)
    client, cluster, temp_dir = setup_dask_client(
        workload_type="cpu",    # CPU-intensive analysis
        reserve_mem_gb=60,      # Reserve memory for OS and I/O
        dashboard=True          # Enable dashboard for monitoring
    )
    
    print(f"📊 Cluster: {len(client.scheduler_info()['workers'])} workers")
    print(f"📁 Temp: {temp_dir}")
    print(f"🖥️  Dashboard: {client.dashboard_link}")
    
    try:
        # Create large synthetic dataset (adjust for your needs)
        print("📂 Creating synthetic dataset...")
        shape = (365*5, 180, 360)  # 5 years of daily global data
        chunks = (365, 90, 180)    # ~512MB chunks
        
        # Simulate climate data
        ds = xr.Dataset({
            'temperature': (['time', 'lat', 'lon'], 
                          da.random.random(shape, chunks=chunks, dtype='float32')),
            'precipitation': (['time', 'lat', 'lon'], 
                            da.random.random(shape, chunks=chunks, dtype='float32'))
        }, coords={
            'time': np.arange(shape[0]),
            'lat': np.linspace(-90, 90, shape[1]),
            'lon': np.linspace(-180, 180, shape[2])
        })
        
        print(f"   Dataset size: {ds.nbytes / (1024**3):.1f} GB")
        
        # Perform analysis
        print("🧮 Running climatological analysis...")
        start_time = time.perf_counter()
        
        # Calculate monthly climatology
        temp_climatology = ds.temperature.groupby('time.month').mean()
        precip_climatology = ds.precipitation.groupby('time.month').mean()
        
        # Compute results
        temp_result = temp_climatology.compute()
        precip_result = precip_climatology.compute()
        
        analysis_time = time.perf_counter() - start_time
        
        print(f"✅ Analysis completed in {analysis_time:.1f} seconds")
        print(f"   Temperature climatology: {temp_result.shape}")
        print(f"   Precipitation climatology: {precip_result.shape}")
        
        # Save results
        output_dir = Path("results")
        output_dir.mkdir(exist_ok=True)
        
        print("💾 Saving results...")
        temp_result.to_netcdf(output_dir / "temperature_climatology.nc")
        precip_result.to_netcdf(output_dir / "precipitation_climatology.nc")
        
        print("🎉 HPC analysis completed successfully!")
        
    except Exception as e:
        print(f"❌ Analysis failed: {e}")
        raise
    
    finally:
        # Clean shutdown
        print("🧹 Shutting down cluster...")
        client.close()
        cluster.close()

if __name__ == "__main__":
    main()
'''

# Save analysis script
analysis_file = output_dir / "analysis.py"
with open(analysis_file, 'w') as f:
    f.write(analysis_script)
analysis_file.chmod(0o755)
print(f"📄 Analysis script saved: {analysis_file}")

print(f"\n🚀 USAGE INSTRUCTIONS:")
print(f"   PBS:   qsub {pbs_file.name}")
print(f"   SLURM: sbatch {slurm_file.name}")
print(f"   Local: python {analysis_file.name}")

print(f"\n📂 All files saved to: {output_dir.absolute()}")

## SSH Tunneling for Dashboard Access

One of the biggest challenges on HPC systems is accessing the Dask dashboard. Let's demonstrate SSH tunneling setup.

In [ ]:
def demonstrate_ssh_tunneling(hostname=None, dashboard_port=8787):
    """Demonstrate SSH tunneling setup for dashboard access."""
    
    print("🔗 SSH TUNNEL SETUP FOR DASHBOARD ACCESS")
    print("=" * 60)
    
    if hostname is None:
        hostname = socket.gethostname()
    
    print(f"🖥️  Compute Node: {hostname}")
    print(f"📡 Dashboard Port: {dashboard_port}")
    
    # Check if we're on a known HPC system
    hpc_systems = {
        'gadi': {
            'login_node': 'gadi.nci.org.au',
            'description': 'NCI Gadi (Australia)',
            'username_format': 'your_username'
        },
        'raijin': {
            'login_node': 'raijin.nci.org.au', 
            'description': 'NCI Raijin (Legacy)',
            'username_format': 'your_username'
        },
        'magnus': {
            'login_node': 'magnus.pawsey.org.au',
            'description': 'Pawsey Magnus (Australia)',
            'username_format': 'your_username'
        },
        'topaz': {
            'login_node': 'topaz.pawsey.org.au',
            'description': 'Pawsey Topaz (Australia)',
            'username_format': 'your_username'
        },
        'summit': {
            'login_node': 'summit.olcf.ornl.gov',
            'description': 'ORNL Summit (USA)',
            'username_format': 'your_username'
        },
        'frontera': {
            'login_node': 'frontera.tacc.utexas.edu',
            'description': 'TACC Frontera (USA)',
            'username_format': 'your_username'
        }
    }
    
    detected_system = None
    login_node = None
    
    for hpc_name, info in hpc_systems.items():
        if hpc_name in hostname.lower():
            detected_system = info
            login_node = info['login_node']
            break
    
    if detected_system:
        print(f"\n🎯 Detected HPC System: {detected_system['description']}")
        print(f"🔗 SSH Tunnel Commands:")
        print(f"   ")
        print(f"   1️⃣  From your LOCAL machine, run:")
        print(f"   ssh -N -L 8787:{hostname}:{dashboard_port} {detected_system['username_format']}@{login_node}")
        print(f"   ")
        print(f"   2️⃣  Then open in your browser:")
        print(f"   http://localhost:8787")
        
        print(f"\n📱 Alternative Method (if direct compute node access blocked):")
        print(f"   Step 1: SSH to login node with port forwarding")
        print(f"   ssh -L 8787:localhost:8787 {detected_system['username_format']}@{login_node}")
        print(f"   ")
        print(f"   Step 2: From login node, tunnel to compute node")
        print(f"   ssh -N -L 8787:{hostname}:{dashboard_port} {hostname}")
        print(f"   ")
        print(f"   Step 3: Open browser on local machine")
        print(f"   http://localhost:8787")
        
    else:
        print(f"\n🔗 Generic SSH Tunnel Command:")
        print(f"   From your local machine:")
        print(f"   ssh -N -L 8787:{hostname}:{dashboard_port} username@your_hpc_login_node")
        print(f"   ")
        print(f"   Then open: http://localhost:8787")
    
    print(f"\n💡 Dashboard Features You'll See:")
    print(f"   • Status: Real-time worker and task information")
    print(f"   • Task Stream: Live view of task execution timeline")
    print(f"   • Memory: Worker memory usage and spill activity")
    print(f"   • Progress: Long-running operation progress bars")
    print(f"   • Profile: Performance profiling and bottleneck analysis")
    print(f"   • Workers: Individual worker status and logs")
    
    print(f"\n⚠️  SSH Tunnel Troubleshooting:")
    print(f"   • Ensure port {dashboard_port} is not blocked by firewalls")
    print(f"   • Try different local port if 8787 is busy: -L 8788:{hostname}:{dashboard_port}")
    print(f"   • Check that your SSH key authentication works")
    print(f"   • Verify compute node network access policies")
    print(f"   • Some systems require VPN connection first")
    
    print(f"\n🔒 Security Notes:")
    print(f"   • SSH tunnels encrypt all dashboard traffic")
    print(f"   • Dashboard is only accessible from your local machine")
    print(f"   • Remember to close tunnels when job completes")
    
    return {
        'hostname': hostname,
        'dashboard_port': dashboard_port,
        'detected_system': detected_system,
        'login_node': login_node
    }


# Demonstrate SSH tunneling
tunnel_info = demonstrate_ssh_tunneling()

print("\n✅ SSH tunneling instructions generated")

## Test Cluster Creation

Let's create a test cluster to verify everything works and demonstrate dashboard access.

In [ ]:
print("🧪 TESTING CLUSTER CREATION")
print("=" * 40)

# Use conservative settings for testing
test_workers = min(4, hpc_env['detected_resources'].get('cpus', 4) // 2)
test_memory = min(20, hpc_env['detected_resources'].get('memory_gb', 40) // 2)

print(f"Creating test cluster with {test_workers} workers, {test_memory} GB reserved")

try:
    # Create cluster
    client, cluster, temp_dir = setup_dask_client(
        workload_type="mixed",
        max_workers=test_workers,
        reserve_mem_gb=test_memory,
        dashboard=True
    )
    
    print(f"\n✅ Cluster created successfully!")
    
    # Display cluster information
    scheduler_info = client.scheduler_info()
    workers = scheduler_info['workers']
    
    print(f"\n📊 Cluster Information:")
    print(f"   Workers: {len(workers)}")
    print(f"   Scheduler: {scheduler_info['address']}")
    print(f"   Dashboard: {client.dashboard_link}")
    print(f"   Temp directory: {temp_dir}")
    
    # Show worker details
    if workers:
        print(f"\n🔧 Worker Details:")
        for worker_id, worker_info in list(workers.items())[:3]:  # Show first 3
            memory_limit_gb = worker_info.get('memory_limit', 0) / (1024**3)
            print(f"   {worker_id[:12]}... | {worker_info['nthreads']} threads | {memory_limit_gb:.1f} GB")
        
        if len(workers) > 3:
            print(f"   ... and {len(workers) - 3} more workers")
    
    # Show SSH tunnel information for this specific cluster
    if client.dashboard_link:
        from urllib.parse import urlparse
        try:
            parsed = urlparse(client.dashboard_link)
            dashboard_port = parsed.port or 8787
            
            print(f"\n🔗 SSH Tunnel for this cluster:")
            hostname = socket.gethostname()
            print(f"   ssh -N -L 8787:{hostname}:{dashboard_port} username@your_login_node")
            print(f"   Then open: http://localhost:8787")
        except Exception:
            print(f"   Dashboard: {client.dashboard_link}")
    
    # Run a quick performance test
    print(f"\n⚡ Running quick performance test...")
    
    with timer("Performance test") as t:
        # Create test computation
        x = da.random.random((2000, 2000), chunks=(500, 500))
        y = da.random.random((2000, 2000), chunks=(500, 500))
        
        # Perform computation
        result = ((x + y) * 2).mean().compute()
    
    print(f"   Test completed in {t['total']:.2f} seconds")
    print(f"   Result: {result:.6f}")
    print(f"   Throughput: {(2 * 2000 * 2000 * 8) / (1024**2) / t['total']:.1f} MB/s")
    
    # Show resource utilization
    print(f"\n📈 Current Resource Utilization:")
    try:
        if PSUTIL_AVAILABLE:
            cpu_percent = psutil.cpu_percent(interval=1)
            memory = psutil.virtual_memory()
            print(f"   CPU: {cpu_percent:.1f}%")
            print(f"   Memory: {memory.used / (1024**3):.1f} GB / {memory.total / (1024**3):.1f} GB ({memory.percent:.1f}%)")
        else:
            print(f"   System monitoring not available")
    except Exception as e:
        print(f"   Could not get system stats: {e}")
    
    # Keep cluster running for a moment
    print(f"\n⏱️  Cluster will remain active for 10 seconds for inspection...")
    print(f"   💡 This is a good time to check the dashboard if you have SSH tunnel set up!")
    time.sleep(10)
    
except Exception as e:
    print(f"❌ Cluster creation failed: {e}")
    client = None
    cluster = None

finally:
    # Clean up
    if 'client' in locals() and client:
        print(f"\n🧹 Shutting down test cluster...")
        client.close()
        cluster.close()
        print(f"   Cluster shut down cleanly")

print(f"\n✅ Cluster test completed")

## Job Monitoring Commands

Let's learn about monitoring jobs and resources on different HPC schedulers.

In [ ]:
def demonstrate_job_monitoring():
    """Show job monitoring commands for PBS and SLURM."""
    
    print("📊 JOB MONITORING COMMANDS")
    print("=" * 60)
    
    # Check current environment
    pbs_job = os.environ.get('PBS_JOBID')
    slurm_job = os.environ.get('SLURM_JOB_ID')
    
    if pbs_job:
        print(f"🐚 PBS JOB MONITORING (Current Job: {pbs_job})")
        print(f"─" * 50)
        
        monitoring_commands = {
            "Job Status": f"qstat -f {pbs_job}",
            "Resource Usage": f"qstat -f {pbs_job} | grep resources_used",
            "Queue Information": "qstat -Q",
            "All Your Jobs": "qstat -u $USER",
            "Node Information": "cat $PBS_NODEFILE",
            "JobFS Usage": "df -h $PBS_JOBFS",
            "Job History": f"tracejob {pbs_job}",
            "Detailed Queue Info": "pbsnodes -a | grep -A5 -B1 'state = free'"
        }
        
        for description, command in monitoring_commands.items():
            print(f"   {description:20}: {command}")
        
        # Try to get actual job information
        print(f"\n📈 CURRENT JOB INFORMATION:")
        try:
            result = subprocess.run(
                ['qstat', '-f', pbs_job], 
                capture_output=True, text=True, timeout=10
            )
            if result.returncode == 0:
                # Parse interesting fields
                lines = result.stdout.split('\n')
                relevant_lines = []
                for line in lines:
                    if any(keyword in line.lower() for keyword in 
                          ['job_name', 'queue', 'resources_used.walltime', 
                           'resources_used.mem', 'job_state', 'exec_host']):
                        relevant_lines.append(line.strip())
                
                for line in relevant_lines[:10]:  # Show first 10 relevant lines
                    print(f"   {line}")
            else:
                print(f"   qstat command failed (not available or job not found)")
        except Exception as e:
            print(f"   Could not run qstat: {e}")
            
    elif slurm_job:
        print(f"🔰 SLURM JOB MONITORING (Current Job: {slurm_job})")
        print(f"─" * 50)
        
        monitoring_commands = {
            "Job Status": f"squeue -j {slurm_job}",
            "Detailed Job Info": f"scontrol show job {slurm_job}",
            "Resource Usage": f"sstat -j {slurm_job} --format=AveCPU,AveRSS,MaxRSS",
            "Job Efficiency": f"seff {slurm_job}",
            "All Your Jobs": "squeue -u $USER",
            "Partition Info": "sinfo",
            "Node Information": f"squeue -j {slurm_job} -o %N",
            "Job History": f"sacct -j {slurm_job} --format=JobID,JobName,Partition,Account,AllocCPUS,State,ExitCode"
        }
        
        for description, command in monitoring_commands.items():
            print(f"   {description:20}: {command}")
        
        # Try to get actual job information
        print(f"\n📈 CURRENT JOB INFORMATION:")
        try:
            result = subprocess.run(
                ['squeue', '-j', slurm_job, '-o', '%i,%j,%t,%M,%l,%D,%C'], 
                capture_output=True, text=True, timeout=10
            )
            if result.returncode == 0:
                lines = result.stdout.strip().split('\n')
                if len(lines) >= 2:
                    headers = lines[0].split(',')
                    values = lines[1].split(',')
                    for header, value in zip(headers, values):
                        print(f"   {header}: {value}")
                else:
                    print(f"   Job info: {result.stdout}")
            else:
                print(f"   squeue command failed (not available or job not found)")
        except Exception as e:
            print(f"   Could not run squeue: {e}")
    
    else:
        print(f"💻 LOCAL SYSTEM MONITORING")
        print(f"─" * 30)
        
        monitoring_commands = {
            "Interactive Process Viewer": "htop",
            "Memory Usage": "free -h",
            "Disk Usage": "df -h",
            "System Load": "uptime",
            "Process Tree": "pstree -p",
            "Python Processes": "ps aux | grep python",
            "Dask Dashboard": "http://localhost:8787 (if enabled)"
        }
        
        for description, command in monitoring_commands.items():
            print(f"   {description:25}: {command}")
    
    # General monitoring tips
    print(f"\n💡 MONITORING BEST PRACTICES:")
    print(f"   • Check resource usage regularly during long jobs")
    print(f"   • Monitor memory usage to detect spilling")
    print(f"   • Watch for worker failures or restarts")
    print(f"   • Use dashboard task stream to identify bottlenecks")
    print(f"   • Set up job notifications for completion/failure")
    
    print(f"\n🚨 WARNING SIGNS TO WATCH FOR:")
    print(f"   • Memory usage near limits (>90%)")
    print(f"   • High I/O wait times")
    print(f"   • Frequent worker restarts")
    print(f"   • Tasks stuck in pending state")
    print(f"   • Unusual network traffic patterns")

# Demonstrate monitoring
demonstrate_job_monitoring()

## HPC Best Practices

Let's summarize the key best practices for running dask_setup on HPC systems.

In [ ]:
def print_hpc_best_practices():
    """Print comprehensive HPC best practices for dask_setup."""
    
    print("\n" + "=" * 60)
    print("💡 HPC BEST PRACTICES FOR DASK_SETUP")
    print("=" * 60)
    
    print("""
🎯 RESOURCE REQUEST STRATEGIES:

   Memory Planning:
   • Request 20-40% more memory than data size
   • Account for intermediate results and overhead
   • Consider: data_size × 2-3 for safety margin
   
   CPU Selection:
   • Fewer workers with more memory often better
   • Consider: cores = memory_gb / 4-8 (for balanced jobs)
   • I/O-heavy: More workers, less memory each
   • CPU-heavy: Fewer workers, more memory each
   
   Example Resource Requests:
   • Small job: 24 cores, 100 GB, 2 hours
   • Medium job: 48 cores, 200 GB, 6 hours
   • Large job: 96 cores, 400 GB, 12 hours

🖥️  SCHEDULER-SPECIFIC OPTIMIZATIONS:

   PBS/Torque (NCI Gadi, etc.):
   • Use $PBS_JOBFS for fast local storage
   • Set TMPDIR="$PBS_JOBFS" in job script
   • Monitor jobfs usage: df -h $PBS_JOBFS
   • Choose appropriate queue (normal, express, hugemem)
   
   SLURM (Many university clusters):
   • Create temp directory: TMPDIR="/tmp/slurm_$SLURM_JOB_ID"
   • Use --mem vs --mem-per-cpu appropriately
   • Consider --exclusive for memory-intensive jobs
   • Monitor with sstat and seff

🔗 DASHBOARD ACCESS STRATEGIES:

   SSH Tunneling (Most Common):
   • From local: ssh -N -L 8787:compute_node:8787 login_node
   • Then browse: http://localhost:8787
   • Use different ports if 8787 busy: -L 8788:...
   
   VNC/X11 Forwarding (Alternative):
   • ssh -X username@login_node
   • Run browser on login node
   • Higher latency but sometimes easier
   
   JupyterHub Integration:
   • Some systems provide Jupyter with port forwarding
   • Check if your HPC offers this service

💾 STORAGE AND I/O OPTIMIZATION:

   Fast Local Storage:
   • Use $PBS_JOBFS, /tmp, or /dev/shm for spilling
   • Avoid network filesystems for temporary data
   • Monitor disk usage in job scripts
   
   Data Placement:
   • Keep input data on fast storage during analysis
   • Use appropriate chunking for network filesystems
   • Consider data locality for multi-node jobs
   
   Output Management:
   • Write results to network storage at end
   • Use compression for large outputs
   • Clean up temporary files in job script

🔧 CONFIGURATION EXAMPLES:

   Memory-Intensive Analysis:
   ```python
   client, cluster, temp_dir = setup_dask_client(
       workload_type="cpu",
       max_workers=2,              # Fewer workers
       reserve_mem_gb=80           # More memory reserved
   )
   ```
   
   I/O-Heavy Processing:
   ```python
   client, cluster, temp_dir = setup_dask_client(
       workload_type="io",
       max_workers=8,              # More workers
       reserve_mem_gb=40           # Less memory per worker
   )
   ```
   
   Balanced Scientific Computing:
   ```python
   client, cluster, temp_dir = setup_dask_client(
       workload_type="mixed",      # Auto-detect optimal settings
       reserve_mem_gb=60           # Standard reservation
   )
   ```

📊 MONITORING AND DEBUGGING:

   Real-Time Monitoring:
   • Always enable dashboard for development: dashboard=True
   • Use SSH tunnels to access dashboard remotely
   • Monitor task stream for bottlenecks
   • Watch memory usage and spilling activity
   
   Job-Level Monitoring:
   • PBS: qstat -f $PBS_JOBID (resource usage)
   • SLURM: seff $SLURM_JOB_ID (efficiency report)
   • Check system logs for worker failures
   • Monitor network and storage I/O
   
   Performance Profiling:
   • Use client.profile() for detailed analysis
   • Identify compute vs I/O bound sections
   • Optimize chunk sizes based on profiling

🚨 COMMON PITFALLS TO AVOID:

   Resource Issues:
   • Don't request more resources than you can use
   • Avoid memory oversubscription (>90% usage)
   • Don't ignore spill directory disk usage
   • Account for Python overhead (2-4 GB per worker)
   
   Network Issues:
   • Avoid putting temporary data on network storage
   • Don't assume high bandwidth between compute nodes
   • Test dashboard access before long jobs
   • Be aware of firewall restrictions
   
   Job Management:
   • Always include error handling in job scripts
   • Set up notification emails for job completion
   • Plan for job time limits and checkpointing
   • Clean up resources in finally blocks

📋 EXAMPLE JOB WORKFLOW:

   1. Development Phase:
      • Test on small datasets locally or interactively
      • Optimize chunking and memory settings
      • Verify dashboard access works
   
   2. Production Submission:
      • Generate appropriate job script
      • Request resources with safety margin
      • Include monitoring and cleanup code
   
   3. Job Monitoring:
      • Set up SSH tunnel for dashboard
      • Monitor resource usage periodically
      • Check for errors in job output
   
   4. Results and Cleanup:
      • Verify output data integrity
      • Archive or move results to permanent storage
      • Clean up temporary files and directories
""")

# Display best practices
print_hpc_best_practices()

## Conclusion

You've now learned how to effectively integrate dask_setup with HPC systems! Here's what we covered:

### Key Takeaways:
1. **Environment Detection**: Automatic recognition of PBS/SLURM environments and resource extraction
2. **Job Script Generation**: Automated creation of optimized job scripts for different schedulers
3. **SSH Tunneling**: Secure dashboard access from local machines to HPC compute nodes
4. **Resource Optimization**: Smart resource allocation based on workload characteristics
5. **Monitoring Integration**: Comprehensive job and system monitoring strategies

### HPC Integration Benefits:
- **Seamless Setup**: dask_setup automatically detects and uses HPC resources
- **Optimized Performance**: Fast local storage ($PBS_JOBFS) for spilling and temporary data
- **Remote Monitoring**: SSH tunneling enables dashboard access from anywhere
- **Resource Efficiency**: Intelligent worker configuration based on allocated resources
- **Error Recovery**: Proper cleanup and error handling in job scripts

### Files Generated:
All the job scripts and examples have been saved to the `hpc_scripts/` directory:
- `dask_analysis.pbs` - PBS job script for systems like NCI Gadi
- `dask_analysis.slurm` - SLURM job script for university clusters  
- `analysis.py` - Example dask_setup analysis script

### Next Steps:
- Customize the job scripts for your specific HPC system and project
- Test the SSH tunneling setup with a small job first
- Adapt the analysis script template for your scientific workflow
- Set up monitoring and notification systems for production jobs
- Explore advanced features like multi-node clusters and specialized queues

### Quick Start Commands:
```bash
# Submit PBS job
qsub dask_analysis.pbs

# Submit SLURM job  
sbatch dask_analysis.slurm

# Set up SSH tunnel (replace with your details)
ssh -N -L 8787:compute_node:8787 username@hpc_login_node
```

Happy HPC computing with dask_setup! 🖥️🚀